# Time-window panel summary (single patient)

4-panel summary per time window: corr matrix, dendrogram, specific heat, network.

**Legacy notebooks merged:**
- TEST_per_patient_time_windows.ipynb

In [ ]:
%matplotlib inline
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname="lrg_eegfc")
from lrg_eegfc.notebook import *

In [ ]:
from lrgsglib.core import compute_laplacian_properties, compute_normalized_linkage, compute_optimal_threshold, get_giant_component_leftoff
from lrgsglib.utils.basic.signals import bandpass_sos
from scipy.cluster.hierarchy import dendrogram, fcluster
from scipy.spatial.distance import squareform
import networkx as nx

patient = list_patients(Path('data/stereoeeg_patients'))[0]
phase = PHASE_LABELS[0]
band = BRAIN_BANDS_NAMES[2]

recording = load_patient_dataset_robust(patient, Path('data/stereoeeg_patients'), phases=[phase])[phase]
timeseries = recording.timeseries
fs = float(recording.parameters.get('fs'))

window_sec = 10.0
overlap = 0.25
window_len = int(window_sec * fs)
step = int(window_len * (1.0 - overlap))
indices = list(range(0, timeseries.shape[1] - window_len + 1, step))[:4]

low, high = BRAIN_BANDS[band]

for idx, start in enumerate(indices):
    window = timeseries[:, start:start + window_len]
    filtered = bandpass_sos(window, low, high, fs, 4)
    corr = build_corr_network(filtered, filter_type='abs', zero_diagonal=True)
    corr[corr < 0.9] = 0
    G = nx.from_numpy_array(corr)
    Gcc, removed = get_giant_component_leftoff(G)

    spect, L, rho, Trho, tau = compute_laplacian_properties(Gcc, tau=None)
    dists = squareform(Trho)
    lnkgM, label_list, _ = compute_normalized_linkage(dists, Gcc, method='ward')
    clTh, *_ = compute_optimal_threshold(lnkgM, scaling_factor=0.98)

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    axes = axes.flatten()

    axes[0].imshow(corr, cmap='viridis')
    axes[0].set_title('Correlation matrix')

    dendrogram(lnkgM, ax=axes[1], color_threshold=clTh)
    axes[1].axhline(clTh, color='red', linestyle='--')
    axes[1].set_yscale('log')
    axes[1].set_title('Dendrogram')

    axes[2].plot(tau[1:], spect, color='blue')
    axes[2].set_xscale('log')
    axes[2].set_title('Specific heat')

    pos = nx.spring_layout(Gcc, seed=42)
    nx.draw(Gcc, pos=pos, ax=axes[3], node_size=30, with_labels=False)
    axes[3].set_title('Network (giant component)')

    fig.suptitle(f'Window {idx} {band} {phase}')
    plt.tight_layout()
    plt.show()